# Characterising a location without knowing its electrode

The question: *what does the network do at one particular station, whatever
electrode is or will be installed there?*

The obvious approach is to bracket it: compute once with no electrode at all
and once with an ideal one. That is right, and it goes further. Adding an
electrode at bus $b$ is a **rank-one change** of the nodal admittance matrix,
so by the Sherman–Morrison formula every nodal voltage is a **Möbius
function** of the electrode admittance $Y_B$:

$$
\underline{u}(Y_B) = \underline{u}_0 - \frac{Y_B\,\underline{z}\,u_{0,b}}{1 + Y_B Z_\text{net}} .
$$

Three quantities that do not depend on the electrode determine the response
to *every* electrode, and two solves give them:

| quantity | meaning |
|---|---|
| $\underline{u}_0$ | the potentials with the electrode at $b$ removed — the open-circuit response of the location |
| $\underline{z}$ | the potential everywhere per ampere injected at $b$, source-free |
| $Z_\text{net} = z_b$ | the driving-point impedance of everything **except** the local electrode |

The two extremes are then not two isolated samples but the **end points of a
curve** that is known in closed form. `bus_response` computes it; this
notebook checks it against solved networks and against a nodal solve written
here in NumPy. Background:
[Characterising a location without its electrode](../../explanation/decomposition/#characterising-a-location-without-its-electrode).

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import groundinsight as gi

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(170)
print(f"groundinsight {gi.__version__}")

## The network

Four stations on a radial 20 kV cable, 1 km per section: the infeed at `S0`,
then `S1`, the location under study `site` with the earth fault, and `S3`
behind it. Every station has a $10\ \Omega$ electrode; the cable is that of
[Which electrode does the faulted station need?](../cuts_and_rho_f_sweep/),
with $k = Z_m/Z_s = 0.5$. The source current has a fifth harmonic.

In [ ]:
FREQUENCIES = [50.0, 250.0]
SOURCE = {50.0: 1000.0, 250.0: 200.0}   # A
Z_STATION = 10.0                        # Ohm
BUSES = ["S0", "S1", "site", "S3"]
SECTIONS = [("C1", "S0", "S1"), ("C2", "S1", "site"), ("C3", "site", "S3")]


def zs(f):
    "Shield-earth loop of one 1 km section in Ohm."
    return 0.1 + 0.2j * f / 50


def zm(f):
    "Coupling of one 1 km section in Ohm."
    return 0.05 + 0.1j * f / 50


CABLE = gi.BranchType(name="cable", grounding_conductor=True,
                      self_impedance_formula="(0.1 + 0.2*j*f/50)*l",
                      mutual_impedance_formula="(0.05 + 0.1*j*f/50)*l",
                      phase_impedance_formula="(0.06 + 0.11*j*f/50)*l")


def network(site_electrode=repr(Z_STATION)):
    "The feeder; ``site_electrode`` is the impedance formula at the site."
    net = gi.create_network(name="feeder", frequencies=FREQUENCIES)
    for bus in BUSES:
        formula = site_electrode if bus == "site" else repr(Z_STATION)
        bus_type = gi.BusType(name=f"electrode_{bus}", system_type="MV", voltage_level=20.0,
                              impedance_formula=formula)
        gi.create_bus(name=bus, type=bus_type, network=net)
    for name, a, b in SECTIONS:
        gi.create_branch(name=name, type=CABLE, from_bus=a, to_bus=b, length=1.0, network=net)
    gi.create_source(name="infeed", bus="S0", values=SOURCE, network=net)
    gi.create_fault(name="earth_fault", bus="site", network=net)
    return net


net = network()
gi.run_fault(net, fault="earth_fault")
response = gi.bus_response(net, fault="earth_fault")
for f in FREQUENCIES:
    z = response.z_network[f]
    print(f"Z_net at {f:5.1f} Hz = {z.real:.6f} {z.imag:+.6f}j Ohm   |Z_net| = {abs(z):.6f} Ohm")

### The second method: a nodal solve by hand

The earthing network of a radial feeder is a ladder, and with the same $k$
on every section it is driven by $(1-k)\,I$, entering at the infeed and
leaving at the fault (in the sign frame of `groundinsight`). The site
electrode is a parameter: `None` for none at all, `0` for an ideal one, which
pins the site to remote earth.

In [ ]:
def by_hand(f, z_site):
    "Earth potentials of all buses for a site electrode z_site (None: open, 0: ideal)."
    index = {bus: i for i, bus in enumerate(BUSES)}
    Y = np.zeros((4, 4), dtype=complex)
    for bus in BUSES:
        if bus != "site":
            Y[index[bus], index[bus]] += 1 / Z_STATION
        elif z_site is not None and z_site != 0:
            Y[index[bus], index[bus]] += 1 / z_site
    for _, a, b in SECTIONS:
        y = 1 / zs(f)
        i, j = index[a], index[b]
        Y[i, i] += y
        Y[j, j] += y
        Y[i, j] -= y
        Y[j, i] -= y
    current = (1 - zm(f) / zs(f)) * SOURCE[f]
    rhs = np.zeros(4, dtype=complex)
    rhs[index["S0"]], rhs[index["site"]] = current, -current
    if z_site == 0:                     # ideal: the site is remote earth
        keep = [index[bus] for bus in BUSES if bus != "site"]
        u = np.zeros(4, dtype=complex)
        u[keep] = np.linalg.solve(Y[np.ix_(keep, keep)], rhs[keep])
    else:
        u = np.linalg.solve(Y, rhs)
    return {bus: u[index[bus]] for bus in BUSES}


def ladder_z_net(f):
    "Closed form: towards the infeed (S1 behind S0) in parallel with towards S3."
    def par(a, b):
        return a * b / (a + b)
    return par(zs(f) + par(Z_STATION, zs(f) + Z_STATION), zs(f) + Z_STATION)


for f in FREQUENCIES:
    print(f"{f:5.1f} Hz  Z_net closed form {ladder_z_net(f):.6f} Ohm,"
          f" deviation {abs(response.z_network[f] - ladder_z_net(f)) / abs(ladder_z_net(f)):.1e}")

## 1  The bracket

`extremes()` evaluates three electrodes: `open` (none at all), `ideal`
($Z_B = 0$) and `worst_passive`, the electrode that maximises the
driving-point impedance among all passive ones. The last one is not the open
end: a purely reactive electrode that cancels the susceptance of the network,
$Y_B = -j\,\operatorname{Im}(1/Z_\text{net})$, lies slightly above it.

In [ ]:
extremes = response.extremes()
extremes.select("case", "frequency_Hz", "Z_bus_Ohm", "Z_network_Ohm", "Z_driving_point_Ohm",
                "EPR_V", "I_electrode_A", "r_epr", "r_current", "EPR_S0_V")

In [ ]:
worst = 0.0
for f in FREQUENCIES:
    y_net = 1 / response.z_network[f]
    z_worst = 1 / complex(0.0, -y_net.imag)
    cases = {"open": None, "ideal": 0, "worst_passive": z_worst}
    for case, z_site in cases.items():
        row = extremes.filter((pl.col("case") == case) & (pl.col("frequency_Hz") == f)).row(0, named=True)
        u = by_hand(f, z_site)
        worst = max(worst, abs(row["EPR_V"] - abs(u["site"])) / abs(u["S0"]),
                    abs(row["EPR_S0_V"] - abs(u["S0"])) / abs(u["S0"]))
    row = extremes.filter((pl.col("case") == "worst_passive") & (pl.col("frequency_Hz") == f)).row(0, named=True)
    print(f"{f:5.1f} Hz  worst passive |Z_dp| = 1/Re(1/Z_net) = {1 / y_net.real:.6f} Ohm"
          f" (table {row['Z_driving_point_Ohm']:.6f}), {1 / y_net.real / abs(1 / y_net) - 1:.3%} above |Z_net|")
print(f"\nlargest deviation of EPR and EPR_S0 from the nodal solve by hand: {worst:.1e}")

ratio = {f: extremes.filter((pl.col("case") == "ideal") & (pl.col("frequency_Hz") == f))["EPR_S0_V"][0]
         / extremes.filter((pl.col("case") == "open") & (pl.col("frequency_Hz") == f))["EPR_S0_V"][0]
         for f in FREQUENCIES}
print("EPR at S0, ideal over open electrode at the site:", {f: round(r, 3) for f, r in ratio.items()})

What the bracket says:

* **$Z_\text{driving point}$** runs from $Z_\text{net}$ (no electrode) to zero
  (ideal electrode). The passive worst case lies 0.08 % above
  $Z_\text{net}$ at 50 Hz and 1.8 % at 250 Hz, where the network is more
  inductive — small in a cable network, but it is the true bound rather than
  an assumed one.
* **`r_epr` does not move.** The closed form says why: the potential of the
  bus is $u_b = u_{0,b}/(1 + Y_B Z_\text{net})$, with and without the
  coupling alike, so the factor cancels in the quotient. The invariance is
  algebra, not an observation.
* **`EPR_S0_V`**: an ideal electrode at the site almost **doubles** the earth
  potential rise at the infeed station. Earthing the site better pushes
  potential onto its neighbours.

## 2  The closed form against solved networks

Two solves, any number of electrodes: here the closed form is checked against
networks that were actually built with that electrode and solved, and against
the nodal solve by hand.

In [ ]:
ELECTRODES = [0.05, 1.0, 10.0, 100.0, complex(4, 9)]
rows = []
for z_b in ELECTRODES:
    formula = repr(z_b) if isinstance(z_b, float) else f"{z_b.real!r} + {z_b.imag!r}*j"
    solved = network(site_electrode=formula)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")          # kept in result.diagnostics
        result = gi.run_fault(solved, fault="earth_fault")
    key = solved.res_fault("earth_fault").row(0, named=True)     # 50 Hz
    closed = response.evaluate(z_b).filter(pl.col("frequency_Hz") == 50.0).row(0, named=True)
    hand = abs(by_hand(50.0, z_b)["site"])
    rows.append({
        "Z_B_Ohm": str(z_b),
        "EPR closed form_V": closed["EPR_V"],
        "EPR solved_V": key["EPR_fault_V"],
        "EPR by hand_V": hand,
        "largest rel. deviation": max(abs(closed["EPR_V"] - key["EPR_fault_V"]),
                                      abs(closed["EPR_V"] - hand)) / hand,
        "r_current closed form": closed["r_current"],
        "r_current solved": key["r_current"],
        "diagnostics": ", ".join(sorted({d.code for d in result.diagnostics})) or "-",
    })
pl.DataFrame(rows)

The closed form reproduces the solved networks to rounding, the earth
potential rise and the reduction factor on a current basis alike. For the
inductive electrode $4 + 9j\ \Omega$ the solver adds `EARTH_GROUP_SPREAD`:
the electrode currents of the stations point in clearly different
directions, so the earth current behind `r_current` describes the dominant
direction only
([Reduction factor on a current basis](../../explanation/reduction-factors/#reduction-factor-on-a-current-basis)).

## 3  The whole curve, without another solve

`sweep` evaluates the closed form for a list of electrodes — a hundred points
cost as much as one. For a real electrode resistance the driving-point
impedance falls monotonically from $Z_\text{net}$ to zero.

In [ ]:
values = list(np.logspace(-2, 3, 101))
curve = response.sweep(values).filter(pl.col("frequency_Hz") == 50.0)
z_dp = curve["Z_driving_point_Ohm"].to_numpy()
print("Z_dp monotonic in Z_B:", bool(np.all(np.diff(z_dp) > 0)))

bracket = extremes.filter(pl.col("frequency_Hz") == 50.0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.semilogx(values, z_dp, label="$|Z_{dp}|$ for $Z_B$ real")
for case, style in (("open", "--"), ("worst_passive", ":")):
    ax1.axhline(bracket.filter(pl.col("case") == case)["Z_driving_point_Ohm"][0], ls=style,
                color="grey", label=case)
ax1.set_xlabel("electrode at the site $Z_B$ / Ohm")
ax1.set_ylabel("driving-point impedance / Ohm")
ax1.legend()
ax2.semilogx(values, curve["EPR_V"], label="EPR at the site")
ax2.semilogx(values, curve["EPR_S0_V"], label="EPR at S0")
ax2.set_xlabel("electrode at the site $Z_B$ / Ohm")
ax2.set_ylabel("earth potential rise at 50 Hz / V")
ax2.legend()
fig.tight_layout()
plt.show()

## 4  What the bracket covers — and what it does not

`open`, `ideal` and `worst_passive` bracket the quantities **of the location
itself**: its earth potential rise and its driving-point impedance. They do
not bracket the **transfer quantities** in the same table — the potential at
a neighbouring station or `r_current`. A purely capacitive electrode is
passive and therefore inside the same class, yet it can drive the potential
at the infeed above the value of the ideal electrode:

In [ ]:
reactances = np.logspace(-2, 3, 401)
capacitive = response.sweep([complex(0.0, -x) for x in reactances]).filter(pl.col("frequency_Hz") == 50.0)
best = capacitive.sort("EPR_S0_V", descending=True).row(0, named=True)
ideal_s0 = bracket.filter(pl.col("case") == "ideal")["EPR_S0_V"][0]
x_best = reactances[int(np.argmax(capacitive["EPR_S0_V"].to_numpy()))]

print(f"EPR at S0 with an ideal electrode at the site:      {ideal_s0:9.3f} V")
print(f"EPR at S0 with Z_B = -j*{x_best:.3f} Ohm at the site:   {best['EPR_S0_V']:9.3f} V"
      f"   by hand {abs(by_hand(50.0, complex(0.0, -x_best))['S0']):9.3f} V")
print(f"electrode current at the site:                      {best['I_electrode_A']:9.3f} A"
      f"   (ideal electrode {bracket.filter(pl.col('case') == 'ideal')['I_electrode_A'][0]:.3f} A)")
print(f"EPR at the site itself for that electrode:          {best['EPR_V']:9.3f} V"
      f"   (bracket: 0 to {bracket.filter(pl.col('case') == 'worst_passive')['EPR_V'][0]:.3f} V)")
print()
r_ideal = bracket.filter(pl.col("case") == "ideal")["r_current"][0]
print(f"largest r_current over these capacitive electrodes: {capacitive['r_current'].max():.6f}"
      f"   (ideal electrode {r_ideal:.6f}, open {bracket.filter(pl.col('case') == 'open')['r_current'][0]:.6f})")

The capacitive electrode cancels the reactance of $Z_\text{net}$, a series
resonance: the electrode current is $u_{0,b}/(Z_\text{net} + Z_B)$, and with
$Z_B = -j\operatorname{Im} Z_\text{net}$ it exceeds the current into an ideal
electrode. The local earth potential rise stays inside its bracket; the
potential at the infeed and `r_current` leave theirs — only slightly here,
but a bound that a passive electrode can cross is no bound. Read the extremes
as bounds for the local quantities only.

## Summary

1. **$Z_\text{net}$ is the number that describes the location** without
   knowing anything about its electrode: what the rest of the earthing
   system contributes in parallel to it.
2. **The extremes bracket the local quantities.** Every passive electrode
   gives an earth potential rise and a driving-point impedance between
   `ideal` and `worst_passive`; if even the worst passive case keeps the
   limits, the location needs no further field computation for that
   question.
3. **The curve in between is known in closed form.** A sensitivity study
   over electrode characteristics needs no further solve — only `evaluate`
   or `sweep`.
4. **Two reduction factors, two statements.** `r_epr` characterises the
   cable run and does not respond to the electrode at the location;
   `r_current` describes how the fault current divides there.

[Which electrode does the faulted station need?](../cuts_and_rho_f_sweep/)
splits $Z_\text{net}$ further into its directions with `analyze_cuts`.
The API: [Location response](../../api/response/).